# The top!

In [ ]:
import mne
from anlffr.helper import biosemi2mne as bs
from anlffr.preproc import find_blinks
from matplotlib import pyplot as plt
from mne import compute_proj_epochs
from mne import concatenate_epochs
import pylab as pl
import numpy as np
import os
import fnmatch
from scipy.fft import fft
from scipy.signal import find_peaks
import gc

In [ ]:
# fpath = "/Users/SHB330/Library/CloudStorage/OneDrive-SharedLibraries-UniversityofPittsburgh/SHRS CSD SNAPLAB - Documents/CORE study/PilotData/SW/Attention_EEG/"
fpath = "/Users/SHB330/Library/CloudStorage/OneDrive-SharedLibraries-UniversityofPittsburgh/SHRS CSD SNAPLAB - Documents/CORE study/PilotData/SW/Attention_EEG/"
subj = 'SP003'
resultsdir = 'results/'

In [ ]:
lefts = []
rights = []
leads = []
lags = []

event_counts = {2: 0, 3: 0, 4: 0, 5: 0, 6: 0}

channels_to_drop = ['GSR1', 'GSR2', 'Erg1', 'Erg2', 'Resp', 'Plet', 'Temp']

bdfs = fnmatch.filter(os.listdir(fpath), subj + '_Attention_Task*.bdf')
if subj == 'HL012':
    bdfs = ['HL012_AttentionTask.bdf', 'HL012_AttentionTask+004.bdf', 'HL012_AttentionTask+005.bdf', 'HL012_AttentionTask+006.bdf', 'HL012_AttentionTask+002.bdf', 'HL012_AttentionTask+003.bdf', 'HL012_AttentionTask+001.bdf']
if subj == 'HL025':
    bdfs = ['HL025_AttentionTask+002.bdf', 'HL025_AttentionTask+003.bdf', 'HL025_AttentionTask+004.bdf']
if subj == 'SP014':
    bdfs = ['SP014_AttentionTask.bdf', 'SP014_AttentionTask+004.bdf', 'SP014_AttentionTask+002.bdf', 'SP014_AttentionTask+003.bdf', 'SP014_AttentionTask+001.bdf']
    ###ABR has been recorded with the same name as Attention - check to make sure it's written somewhere 
print(bdfs)

for k, bdf in enumerate(bdfs):
    if bdf == 'HL008_AttentionTask+004.bdf':
        bdf_fullpath = os.path.join(fpath, bdf)
        raw, eves = bs.importbdf(bdf_fullpath, hptsname='biosemi10_20.hpts')
        total = raw.times[-1]
        end_time = total - 120
        raw = raw.crop(tmin=0, tmax=end_time)
    else:
        bdf_fullpath = os.path.join(fpath, bdf)
        raw, eves = bs.importbdf(bdf_fullpath, hptsname='biosemi10_20.hpts') 
    print(f'Running file: {bdf}')
    raw.notch_filter(np.arange(60, 241, 60), filter_length='auto', fir_design='firwin')
    raw.filter(1, 40)

    # if subj == 'SP004' or 'SP019':
    #     raw.drop_channels(channels_to_drop)

    blinks = find_blinks(raw, ch_name=['Fp1'])
    if subj in ['HR', 'SP004', 'HL004', 'HL007', 'HL010', 'HL022', 'HL031']:
        if k == 0:
            epochs_blinks = mne.Epochs(raw, blinks, event_id=998, baseline=(-0.25, 0.25),
                                       reject=dict(eeg=500e-6), tmin=-0.25, tmax=0.25)
            blink_proj = compute_proj_epochs(epochs_blinks, n_eeg=2)
        
        raw.add_proj(blink_proj)
        #raw.del_proj(1)
        raw.apply_proj()
    
    elif subj == 'HL022':
        if k == 0:
            epochs_blinks = mne.Epochs(raw, blinks, event_id=998, baseline=(-0.25, 0.25),
                                       reject=dict(eeg=500e-6), tmin=-0.25, tmax=0.25)
            blink_proj = compute_proj_epochs(epochs_blinks, n_eeg=4)
        
        raw.add_proj(blink_proj)
        raw.del_proj(1)
        raw.del_proj(2)
        raw.apply_proj()

    else:
        if k == 0:
            epochs_blinks = mne.Epochs(raw, blinks, event_id=998, baseline=(-0.25, 0.25),
                                       reject=dict(eeg=500e-6), tmin=-0.25, tmax=0.25)
            blink_proj = compute_proj_epochs(epochs_blinks, n_eeg=3)
        
        raw.add_proj(blink_proj)
        raw.del_proj(1)
        raw.apply_proj()
        
    for event_id in [2, 3, 4, 5, 6]:
        event_counts[event_id] += np.sum(eves[:, 2] == event_id)

    events_left = [7,8]
    events_left_use = [event_left for event_left in events_left if event_left in eves[:,2]]
    
    events_right = [9,10]
    events_right_use = [event_right for event_right in events_right if event_right in eves[:,2]]

    events_lead = [7,9]
    events_lead_use = [event_lead for event_lead in events_lead if event_lead in eves[:,2]]

    events_lag = [8,10]
    events_lag_use = [event_lag for event_lag in events_lag if event_lag in eves[:,2]]

    if events_left_use:
        epochs_left = mne.Epochs(raw, eves, event_id=events_left_use, baseline=(-0.5, 0.0), proj=True,
                        tmin=-0.5, tmax=5, reject=dict(eeg=200e-6))
    if events_right_use:
        epochs_right = mne.Epochs(raw, eves, event_id=events_right_use, baseline=(-0.5, 0.0), proj=True,
                        tmin=-0.5, tmax=5, reject=dict(eeg=200e-6))
    if events_lead_use:
            epochs_lead = mne.Epochs(raw, eves, event_id=events_lead_use, baseline=(-0.5, 0.0), proj=True,
                            tmin=-0.5, tmax=5, reject=dict(eeg=200e-6))
    if events_lag_use:
            epochs_lag = mne.Epochs(raw, eves, event_id=events_lag_use, baseline=(-0.5, 0.0), proj=True,
                            tmin=-0.5, tmax=5, reject=dict(eeg=200e-6))
    
    lefts += [epochs_left, ]
    rights += [epochs_right, ]
    leads += [epochs_lead, ]
    lags += [epochs_lag, ]


left = concatenate_epochs(lefts)
right = concatenate_epochs(rights)
lead = concatenate_epochs(leads)
lag = concatenate_epochs(lags) 

for event_id, count in event_counts.items():
    print(f"{event_id}: {count} ")

In [ ]:
evoked_lead = lead.average()
evoked_lag = lag.average()

# fname_left = fpath+resultsdir+subj + '_evoked_lead_ave2.fif'
# fname_right = fpath+resultsdir+subj + '_evoked_lag_ave2.fif'

# evoked_lead.save(fname_left,overwrite=True)
# evoked_lag.save(fname_right, overwrite=True)

In [ ]:
raw.plot_projs_topomap()

In [ ]:
evoked_left = left.average()
evoked_right = right.average()

In [ ]:
epoch_left_pre = left.copy().crop(tmin=0.4, tmax=1.6)
epoch_right_pre = right.copy().crop(tmin=0.4, tmax=1.6) 

In [ ]:
# fname_left = fpath+resultsdir+subj + '_evoked_left_ave.fif'
# fname_right = fpath+resultsdir+subj + '_evoked_right_ave.fif'

# evoked_left.save(fname_left,overwrite=True)
# evoked_right.save(fname_right, overwrite=True)

evoked_lead.filter(1, 20)
evoked_lag.filter(1, 20)

x = evoked_lead.data * 1e6
t = evoked_lag.times
y = evoked_lag.data * 1e6
sum_x = np.zeros([evoked_lead.data.shape[1]])
sum_y = np.zeros([evoked_lead.data.shape[1]])
for i in [30,31,4,25]:
    sum_x += x[i,]
    sum_y += y[i,]
z = np.asarray((sum_x/4, sum_y/4))
time = np.linspace(-0.5, 5, y.shape[1])

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(time, z[0], linewidth=1.5, color='red', label='Evoked Lead') 
ax.plot(time, z[1], linewidth=1.5, color='black', label='Evoked Lag')
# ax.axvline(x=0.4, linestyle='--', linewidth=0.75, color='blue')  
for x_val in [2.1, 2.6, 3.1, 3.6, 4.1]:
    ax.axvline(x=x_val, linestyle='--', linewidth=0.75, color='red')
for x_val in [2.35, 2.85, 3.35, 3.85, 4.35]:
    ax.axvline(x=x_val, linestyle='--', linewidth=0.75, color='black')
ax.set_xlabel('Time (s)', fontsize=16)
ax.set_ylabel('Evoked Potential(µV)', fontsize=16)
ax.set_ylim(-4,4)
ax.tick_params(axis='both', labelsize=14)
ax.legend(['Attend Lead', 'Attend Lag'], loc='upper left')
# plt.savefig(foldername+'Evoked_NH_newest.pdf', format='pdf')
plt.show()

In [ ]:
freqs = np.arange(6., 14., 1.)
n_cycles = freqs * 0.4

In [ ]:
power_alpha_left_pre = epoch_left_pre.compute_tfr(method="multitaper", freqs=freqs, picks=['eeg'], return_itc=False, n_cycles=n_cycles)
power_alpha_right_pre = epoch_right_pre.compute_tfr(method="multitaper", freqs=freqs, picks=['eeg'], return_itc=False, n_cycles=n_cycles)

del epoch_right_pre

# fname_pleft_pre = fpath+resultsdir+subj+ '_left_alpha_preprestim_new_tfr.h5'
# fname_pright_pre = fpath+resultsdir+subj+ '_right_alpha_preprestim_new_tfr.h5'

# power_alpha_left_pre.save(fname_pleft_pre, overwrite=True)
# power_alpha_right_pre.save(fname_pright_pre, overwrite=True)

In [ ]:
avg_power_alpha_left_pre = power_alpha_left_pre.data.mean(axis=0).mean(axis=1).mean(axis=1)
avg_power_alpha_right_pre = power_alpha_right_pre.data.mean(axis=0).mean(axis=1).mean(axis=1)

norm_avg_power_alpha_right_left_pre = (avg_power_alpha_right_pre - avg_power_alpha_left_pre)/((avg_power_alpha_left_pre + avg_power_alpha_right_pre)*0.5)*100 

In [ ]:
channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7','P3','Pz','PO3',
 'O1','Oz','O2','PO4','P4','P8','CP6','CP2','C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']
info_picked = mne.pick_info(epoch_left_pre.info, mne.pick_channels(epoch_left_pre.info['ch_names'], include =channels))

fig, ax = plt.subplots(figsize=(8, 6))
im,_ = mne.viz.plot_topomap(norm_avg_power_alpha_right_left_pre, info_picked, names=channels,  axes=ax, ch_type='eeg',show=False)
plt.colorbar(im, ax=ax)
plt.show()

In [ ]:
right= [18,20,21,22,23]
left = [11,9,8,7,6]

central_pre = np.mean(norm_avg_power_alpha_right_left_pre[right] - norm_avg_power_alpha_right_left_pre[left])

In [ ]:
central_pre

In [ ]:
selected_channels = [30, 31, 4, 25]

evoked_lead_crop = evoked_lead.crop(tmin=2.35, tmax=4.35)
evoked_lag_crop = evoked_lag.crop(tmin=2.35, tmax=4.35)

RMS_num = ((evoked_lead_crop.data - evoked_lag_crop.data)**2)
RMS_den = ((evoked_lead_crop.data ** 2 + evoked_lag_crop.data ** 2) / 2)
RMS_central = RMS_num[selected_channels].mean()/RMS_den[selected_channels].mean()

In [ ]:
RMS_central

In [ ]:
print(RMS_central)

## Read and analyze csv

In [ ]:
import pandas as pd

dataset = pd.read_excel('/Users/SHB330/Library/CloudStorage/OneDrive-SharedLibraries-UniversityofPittsburgh/SHRS CSD SNAPLAB - Documents/CORE study/PilotData/SW/Attention_EEG/results/results_Attention.xlsx')
dataset.head

In [ ]:
group1 = dataset.iloc[:14]
group2 = dataset.iloc[14:]


In [ ]:
v1NH = group1["individual_RMS"]
v1HL = group2["individual_RMS"]

In [ ]:
import scipy.stats as stats

plt.figure()
stats.probplot(v1NH, plot=plt)
plt.title("Q–Q plot: NH")
plt.show()

plt.figure()
stats.probplot(v1HL, plot=plt)
plt.title("Q–Q plot: HL")
plt.show()


In [ ]:
t_stat, p_val = stats.ttest_ind(v1NH, v1HL, equal_var=False)

print(f"n1={len(v1NH)}, n2={len(v1HL)}")
print(f"Welch's t = {t_stat:.4f}, p = {p_val:.6g}")
print(f"mean1={v1NH.mean():.4f}, mean2={v1HL.mean():.4f}")
print(f"sd1={v1NH.std(ddof=1):.4f}, sd2={v1HL.std(ddof=1):.4f}")

In [ ]:
import numpy as np

def jackknife_sem(x):
    x = np.asarray(x)
    n = len(x)

    loo_means = np.array([
        np.mean(np.delete(x, i)) for i in range(n)
    ])

    mean_loo = np.mean(loo_means)

    sem = np.sqrt((n - 1) / n * np.sum((loo_means - mean_loo)**2))
    return sem

In [ ]:
sem_v1NH = jackknife_sem(v1NH)
sem_v1HL = jackknife_sem(v1HL)

print("Jackknife SEM g1:", sem_v1NH)
print("Jackknife SEM g2:", sem_v1HL)


In [ ]:
# import numpy as np

# # Hedges' g (bias-corrected Cohen's d)
# n1, n2 = len(v1NH), len(v1HL)
# s1, s2 = v1NH.var(ddof=1), v1HL.var(ddof=1)
# sp = np.sqrt(((n1 - 1)*s1 + (n2 - 1)*s2) / (n1 + n2 - 2))
# d = (v1NH.mean() - v1HL.mean()) / sp
# J = 1 - (3 / (4*(n1 + n2) - 9))  # small sample correction
# g = J * d

# print(f"Hedges' g = {g:.4f}")

In [ ]:
v2NH = group1["Alpha_assymetry_pre"]
v2HL = group2["Alpha_assymetry_pre"]

plt.figure()
stats.probplot(v1NH, plot=plt)
plt.title("Q–Q plot: NH")
plt.show()

plt.figure()
stats.probplot(v1HL, plot=plt)
plt.title("Q–Q plot: HL")
plt.show()

In [ ]:
t_stat, p_val = stats.ttest_ind(v2NH, v2HL, equal_var=False)

print(f"n1={len(v2NH)}, n2={len(v2HL)}")
print(f"Welch's t = {t_stat:.4f}, p = {p_val:.6g}")
print(f"mean21={v2NH.mean():.4f}, mean22={v2HL.mean():.4f}")
print(f"sd1={v2NH.std(ddof=1):.4f}, sd2={v2HL.std(ddof=1):.4f}")


In [ ]:
x_labels = ["Normal Hearing", "Hearing Loss"]
x_positions = np.arange(len(x_labels))

# Example values (replace with your actual data)
mean_changes = [v1NH.mean(), v1HL.mean()]
sem_values = [sem_v1NH, sem_v1HL]


plt.figure(figsize=(7, 5))
bars = plt.bar(x_positions, mean_changes, yerr=sem_values, capsize=5, color='gray', edgecolor='black')

plt.xticks(x_positions, x_labels)
plt.ylabel("Change in Energy", fontsize=14)
# plt.yticks(np.arange(60, 100, 5))
# plt.ylim(60, 100)
plt.tick_params(axis='x', labelsize=12)
plt.tick_params(axis='y', labelsize=12)
plt.title("Attentional Modulation of Evoked Responses", fontsize=16)

# plt.savefig(foldername + 'RMS_evoked_bar.pdf', format='pdf')
plt.tight_layout()
plt.show()

## Alpha

In [ ]:
sem_v2NH = np.std(v2NH, ddof=1) / np.sqrt(len(v2NH))
sem_v2HL = np.std(v2HL, ddof=1) / np.sqrt(len(v2HL))

print("Classic SEM g1:", sem_v2NH)
print("Classic SEM g2:", sem_v2HL)


In [ ]:
# x_labels = ["Normal Hearing Pre-Stim", "Normal Hearing Dur-Stim", "Hearing Loss Pre-Stim", "Hearing Loss Dur-Stim"] 
x_labels = ["Normal Hearing", "Hearing Loss"] 
x_positions = np.arange(len(x_labels))

# mean_changes = [np.mean(NH_assymetry_PO_pre), np.mean(NH_assymetry_PO_dur),np.mean(HL_assymetry_PO_pre), np.mean(HL_assymetry_PO_dur)]
# mean_changes = [np.mean(NH_assymetry_CT_pre), np.mean(HL_assymetry_CT_pre),np.mean(NH_assymetry_CT_dur), np.mean(HL_assymetry_CT_dur)]
mean_changes = [v2NH.mean(), v2HL.mean()]
# sem_values = [NH_assymetry_PO_pre_sem, NH_assymetry_PO_dur_sem, HL_assymetry_PO_pre_sem, HL_assymetry_PO_dur_sem]
# sem_values = [NH_assymetry_CT_pre_sem, HL_assymetry_CT_pre_sem, NH_assymetry_CT_dur_sem, HL_assymetry_CT_dur_sem]
sem_values = [sem_v2NH, sem_v2HL]

plt.figure(figsize=(5, 5))
plt.errorbar(x_positions, mean_changes, yerr=sem_values, fmt='s', capsize=5, color='black', markersize=8)
plt.xticks(x_positions, x_labels)  
plt.yticks([-2,0,2,4,6,8,10, 12])
plt.tick_params(axis='x', labelsize=12)
plt.tick_params(axis='y', labelsize=12)

plt.ylabel("Alpha Power Asymetry(%)", fontsize=14)
# plt.title("Change in Energy during stimulus")
# plt.savefig(foldername+'alpha_power_assymetry_CT.pdf', format='pdf')
plt.show()

### Median and MAD

In [ ]:
import numpy as np

def mad(x):
    med = np.median(x)
    n = x.shape[0]
    return (np.median(np.abs(x - med))/np.sqrt(n))

med1 = np.median(v2NH)
med2 = np.median(v2HL)

mad1 = mad(v2NH)
mad2 = mad(v2HL)

mad1_scaled = 1.4826 * mad1
mad2_scaled = 1.4826 * mad2

print("Group 1: median =", med1, "MAD =", mad1, "scaled MAD =", mad1)
print("Group 2: median =", med2, "MAD =", mad2, "scaled MAD =", mad2)


In [ ]:
x_labels = ["Normal Hearing", "Hearing Loss"] 
x_positions = np.arange(len(x_labels))

mean_changes = [med1, med2]
sem_values = [mad1, mad2]

plt.figure(figsize=(5, 5))
plt.errorbar(x_positions, mean_changes, yerr=sem_values, fmt='s', capsize=5, color='black', markersize=8)
plt.xticks(x_positions, x_labels)  
plt.yticks([-1,0,1,2,3,4,5])
plt.tick_params(axis='x', labelsize=12)
plt.tick_params(axis='y', labelsize=12)

plt.ylabel("Alpha Power Asymetry(%)", fontsize=14)
# plt.title("Change in Energy during stimulus")
# plt.savefig(foldername+'alpha_power_assymetry_CT.pdf', format='pdf')
plt.show()